# M. extorquens B Vitamin Biosynthesis and Lanthanide C1 Metabolism

*Methylobacterium extorquens* AM1 ([NCBI Taxonomy 272630](https://www.ncbi.nlm.nih.gov/Taxonomy/Browser/wwwtax.cgi?id=272630)) is the model organism for lanthanide-dependent methylotrophy. It carries two methanol dehydrogenases:

- **XoxF**: lanthanide-dependent PQQ methanol dehydrogenase (uses La, Ce, or other rare earth elements)
- **MxaF**: calcium-dependent PQQ methanol dehydrogenase (the classical enzyme)

These represent fundamentally different C1 metabolic modes. This notebook asks:

1. Is *M. extorquens* auxotrophic for any B vitamins? Which biosynthesis pathways are complete vs incomplete?
2. What does that mean for optimal growth medium design?
3. Do xoxF-carrying *Methylobacterium* species show different B vitamin profiles than mxaF-only species?

**Portability**: This notebook uses Spark SQL on BERDL JupyterHub. Queries can be adapted for the KBase MCP REST API for workstation use, though the multi-table joins perform best on Spark.

## 1. Setup

In [ ]:
from berdl_notebook_utils import get_spark_session
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

spark = get_spark_session("MextorquensBvitamin")

## 2. Identify M. extorquens in the pangenome

GTDB retains the name *Methylobacterium* for this lineage, while NCBI recently reclassified some strains as *Methylorubrum*. We search for both.

In [ ]:
spark.sql("""
    SELECT s.GTDB_species, s.gtdb_species_clade_id,
           p.no_genomes, p.no_core, p.no_gene_clusters,
           p.no_aux_genome, p.no_singleton_gene_clusters
    FROM kbase_ke_pangenome.pangenome p
    JOIN kbase_ke_pangenome.gtdb_species_clade s
      ON p.gtdb_species_clade_id = s.gtdb_species_clade_id
    WHERE s.GTDB_species LIKE '%extorquens%'
       OR s.GTDB_species LIKE '%Methylorubrum%'
""").show(truncate=False)

## 3. B vitamin biosynthesis gene survey

We search eggNOG annotations for key genes in six B vitamin biosynthesis pathways. For each pathway, we define the expected biosynthesis genes and their KEGG orthologs.

In [ ]:
B_VITAMIN_PATHWAYS = {
    'B1_thiamine': {
        'kegg_pathway': 'ko00730',
        'key_genes': ['thiC', 'thiD', 'thiE', 'thiG', 'thiL', 'thiM', 'thiO', 'thiS'],
        'key_kos': ['K03147', 'K00788', 'K00946', 'K03149', 'K00948', 'K00878']
    },
    'B2_riboflavin': {
        'kegg_pathway': 'ko00740',
        'key_genes': ['ribA', 'ribB', 'ribC', 'ribD', 'ribE', 'ribH'],
        'key_kos': ['K02858', 'K14652', 'K00793', 'K11752', 'K00794', 'K02857']
    },
    'B6_pyridoxine': {
        'kegg_pathway': 'ko00750',
        'key_genes': ['pdxA', 'pdxB', 'pdxJ', 'pdxH', 'pdxK'],
        'key_kos': ['K00097', 'K06215', 'K03474', 'K00275', 'K00868']
    },
    'B7_biotin': {
        'kegg_pathway': 'ko00780',
        'key_genes': ['bioA', 'bioB', 'bioC', 'bioD', 'bioF', 'bioH'],
        'key_kos': ['K00833', 'K01012', 'K02169', 'K01935', 'K00652', 'K02170']
    },
    'B9_folate': {
        'kegg_pathway': 'ko00790',
        'key_genes': ['folA', 'folB', 'folC', 'folE', 'folK', 'folP'],
        'key_kos': ['K00287', 'K01633', 'K11754', 'K01495', 'K00950', 'K00796']
    },
    'B12_cobalamin': {
        'kegg_pathway': 'ko00860',
        'key_genes': ['cobA', 'cobB', 'cobC', 'cobD', 'cobE', 'cobG', 'cobH',
                       'cobI', 'cobJ', 'cobK', 'cobL', 'cobM', 'cobN', 'cobO',
                       'cobP', 'cobQ', 'cobR', 'cobS', 'cobT', 'cobU', 'cobW',
                       'bluB', 'btuR'],
        'key_kos': ['K00798', 'K02224', 'K02225', 'K02227', 'K02229', 'K02230',
                     'K02231', 'K02232', 'K02233', 'K13786', 'K02236', 'K19221']
    }
}

## 4. Query B vitamin genes for M. extorquens

Search by both gene name (`Preferred_name`) and KEGG pathway membership.

In [ ]:
# Collect all gene names across pathways
all_genes = []
for pathway, info in B_VITAMIN_PATHWAYS.items():
    all_genes.extend(info['key_genes'])
gene_list = "','".join(all_genes)

bvit_annotations = spark.sql(f"""
    SELECT
        e.query_name as gene_cluster_id,
        e.Preferred_name,
        e.EC,
        e.KEGG_ko,
        e.KEGG_Pathway,
        e.Description,
        e.COG_category,
        gc.is_core,
        gc.is_auxiliary,
        gc.is_singleton
    FROM kbase_ke_pangenome.eggnog_mapper_annotations e
    JOIN kbase_ke_pangenome.gene_cluster gc
        ON e.query_name = gc.gene_cluster_id
    JOIN kbase_ke_pangenome.gtdb_species_clade sc
        ON gc.gtdb_species_clade_id = sc.gtdb_species_clade_id
    WHERE sc.GTDB_species LIKE '%extorquens%'
      AND (e.Preferred_name IN ('{gene_list}')
        OR e.KEGG_Pathway LIKE '%00730%'
        OR e.KEGG_Pathway LIKE '%00740%'
        OR e.KEGG_Pathway LIKE '%00750%'
        OR e.KEGG_Pathway LIKE '%00780%'
        OR e.KEGG_Pathway LIKE '%00790%'
        OR e.KEGG_Pathway LIKE '%00860%')
    ORDER BY e.KEGG_Pathway, e.Preferred_name
""")

bvit_pd = bvit_annotations.toPandas()
print(f"Total B vitamin-related gene clusters found: {len(bvit_pd)}")
bvit_pd

## 5. Pathway completeness assessment

Map each found gene to its B vitamin pathway and identify which pathways are complete vs incomplete.

In [ ]:
def classify_bvit_pathway(row):
    """Classify a gene cluster annotation into a B vitamin pathway."""
    name = row.get('Preferred_name', '')
    kegg_pathway = str(row.get('KEGG_Pathway', ''))

    for pathway, info in B_VITAMIN_PATHWAYS.items():
        if name in info['key_genes']:
            return pathway
        if info['kegg_pathway'].replace('ko', '') in kegg_pathway:
            return pathway
    return 'other'

bvit_pd['pathway'] = bvit_pd.apply(classify_bvit_pathway, axis=1)
bvit_classified = bvit_pd[bvit_pd['pathway'] != 'other'].copy()

print("=== B Vitamin Pathway Completeness in M. extorquens ===")
print()
for pathway, info in B_VITAMIN_PATHWAYS.items():
    expected_genes = set(info['key_genes'])
    found_genes = set(bvit_classified[bvit_classified['pathway'] == pathway]['Preferred_name'].unique())
    found_in_expected = found_genes & expected_genes
    missing = expected_genes - found_genes

    completeness = len(found_in_expected) / len(expected_genes) * 100 if expected_genes else 0

    print(f"{pathway}:")
    print(f"  Found: {sorted(found_in_expected)} ({len(found_in_expected)}/{len(expected_genes)})")
    print(f"  Missing: {sorted(missing)}")
    print(f"  Completeness: {completeness:.0f}%")

    pathway_clusters = bvit_classified[bvit_classified['pathway'] == pathway]
    n_core = pathway_clusters['is_core'].sum()
    n_aux = pathway_clusters['is_auxiliary'].sum()
    n_sing = pathway_clusters['is_singleton'].sum()
    print(f"  Core/Auxiliary/Singleton: {n_core}/{n_aux}/{n_sing}")
    print()

## 6. Growth medium implications

Missing biosynthesis genes predict auxotrophies — the organism cannot synthesize that vitamin and must obtain it from the medium. Standard *M. extorquens* AM1 media use methanol as carbon source with mineral salts. Some formulations add trace vitamins.

**Caveat**: Missing annotations may reflect gaps in eggNOG sensitivity rather than true gene absence. Predictions should be validated experimentally.

In [ ]:
MEDIUM_COMPONENTS = {
    'B1_thiamine': 'thiamine HCl',
    'B2_riboflavin': 'riboflavin',
    'B6_pyridoxine': 'pyridoxine HCl',
    'B7_biotin': 'biotin',
    'B9_folate': 'folic acid',
    'B12_cobalamin': 'cyanocobalamin or hydroxocobalamin'
}

print("=== Growth Medium Implications ===")
print()

predicted_supplements = []
for pathway, info in B_VITAMIN_PATHWAYS.items():
    expected_genes = set(info['key_genes'])
    found_genes = set(bvit_classified[bvit_classified['pathway'] == pathway]['Preferred_name'].unique())
    missing = expected_genes - found_genes
    completeness = len(found_genes & expected_genes) / len(expected_genes) * 100

    if completeness < 80:
        supplement = MEDIUM_COMPONENTS.get(pathway, 'unknown')
        predicted_supplements.append((pathway, supplement, f"{completeness:.0f}%"))
        print(f"PREDICTED AUXOTROPHY: {pathway}")
        print(f"  Pathway completeness: {completeness:.0f}%")
        print(f"  Recommended supplement: {supplement}")
        print(f"  Missing genes: {sorted(missing)}")
        print()

if not predicted_supplements:
    print("No clear auxotrophies predicted (all pathways >80% complete)")
    print("Note: this may undercount due to annotation sensitivity")

print()
print("Note: Standard M. extorquens AM1 medium typically includes")
print("methanol as carbon source with mineral salts. Some formulations")
print("add trace vitamins. These predictions should be validated by")
print("growth experiments with/without individual vitamin supplements.")

## 7. xoxF vs mxaF annotation audit

A prior analysis of 150 methylotrophic species revealed that eggNOG sometimes annotates **xoxF** (lanthanide-dependent MDH) as `mxaF` with EC 1.1.2.8 / K00114. The true mxaF should have EC 1.1.2.7 / K14028. We audit all PQQ dehydrogenase annotations for *M. extorquens* to distinguish them.

In [ ]:
mdh_all = spark.sql("""
    SELECT
        e.query_name as gene_cluster_id,
        e.Preferred_name,
        e.EC,
        e.KEGG_ko,
        e.Description,
        e.PFAMs,
        gc.is_core,
        gc.is_auxiliary,
        gc.is_singleton,
        sc.GTDB_species
    FROM kbase_ke_pangenome.eggnog_mapper_annotations e
    JOIN kbase_ke_pangenome.gene_cluster gc
        ON e.query_name = gc.gene_cluster_id
    JOIN kbase_ke_pangenome.gtdb_species_clade sc
        ON gc.gtdb_species_clade_id = sc.gtdb_species_clade_id
    WHERE sc.GTDB_species LIKE '%extorquens%'
      AND (e.Preferred_name IN ('xoxF', 'mxaF', 'mxaI', 'exaA')
        OR e.KEGG_ko LIKE '%K14028%'
        OR e.KEGG_ko LIKE '%K14029%'
        OR e.KEGG_ko LIKE '%K00114%'
        OR e.PFAMs LIKE '%PQQ%')
    ORDER BY sc.GTDB_species, e.Preferred_name, e.EC
""")

mdh_pd = mdh_all.toPandas()
print(f"Total PQQ MDH-related gene clusters in M. extorquens: {len(mdh_pd)}")
print()
print("=== MDH Classification ===")
print("Note: eggNOG sometimes annotates xoxF as 'mxaF' with EC 1.1.2.8 / K00114")
print("True mxaF should have EC 1.1.2.7 / K14028")
print()

for _, row in mdh_pd.iterrows():
    name = str(row['Preferred_name'])
    ec = str(row['EC'])
    kegg = str(row['KEGG_ko'])

    if name == 'xoxF':
        likely = 'xoxF (lanthanide-dependent)'
    elif name == 'mxaF' and 'K14028' in kegg:
        likely = 'mxaF (calcium-dependent, canonical)'
    elif name == 'mxaF' and ('K00114' in kegg or '1.1.2.8' in ec):
        likely = 'POSSIBLE xoxF (misannotated as mxaF, EC 1.1.2.8/K00114)'
    elif name == 'mxaI':
        likely = 'mxaI (small subunit)'
    elif name == 'exaA':
        likely = 'exaA (ethanol dehydrogenase, PQQ family)'
    else:
        likely = f'other PQQ enzyme ({name})'

    print(f"  {row['gene_cluster_id']}: {name} | EC={ec} | KEGG={kegg} | {likely}")
    print(f"    Core={row['is_core']} Aux={row['is_auxiliary']} Singleton={row['is_singleton']}")
    print(f"    PFAMs: {row['PFAMs']}")
    print()

## 8. Cross-species: xoxF/mxaF and B vitamin correlation

Do xoxF-carrying *Methylobacterium* species have different B vitamin biosynthesis profiles than mxaF-only species? If so, this would connect lanthanide biology to vitamin metabolism.

In [ ]:
# Step 1: MDH profile per Methylobacterium species
mdh_by_species = spark.sql("""
    SELECT
        sc.GTDB_species,
        sc.gtdb_species_clade_id,
        SUM(CASE WHEN e.Preferred_name = 'xoxF' THEN 1 ELSE 0 END) as xoxF_clusters,
        SUM(CASE WHEN e.Preferred_name = 'mxaF' THEN 1 ELSE 0 END) as mxaF_clusters
    FROM kbase_ke_pangenome.eggnog_mapper_annotations e
    JOIN kbase_ke_pangenome.gene_cluster gc
        ON e.query_name = gc.gene_cluster_id
    JOIN kbase_ke_pangenome.gtdb_species_clade sc
        ON gc.gtdb_species_clade_id = sc.gtdb_species_clade_id
    WHERE sc.GTDB_species LIKE '%Methylobacterium%'
      AND e.Preferred_name IN ('xoxF', 'mxaF')
    GROUP BY sc.GTDB_species, sc.gtdb_species_clade_id
""")

mdh_species_pd = mdh_by_species.toPandas()
mdh_species_pd['mdh_profile'] = mdh_species_pd.apply(
    lambda r: 'BOTH' if r['xoxF_clusters'] > 0 and r['mxaF_clusters'] > 0
              else ('xoxF_only' if r['xoxF_clusters'] > 0 else 'mxaF_only'),
    axis=1
)
print(f"Methylobacterium species with MDH annotations: {len(mdh_species_pd)}")
print(mdh_species_pd['mdh_profile'].value_counts())

In [ ]:
# Step 2: B vitamin gene counts per Methylobacterium species
all_gene_names = []
for info in B_VITAMIN_PATHWAYS.values():
    all_gene_names.extend(info['key_genes'])
gene_str = "','".join(all_gene_names)

bvit_by_species = spark.sql(f"""
    SELECT
        sc.GTDB_species,
        sc.gtdb_species_clade_id,
        e.Preferred_name,
        COUNT(*) as n_clusters
    FROM kbase_ke_pangenome.eggnog_mapper_annotations e
    JOIN kbase_ke_pangenome.gene_cluster gc
        ON e.query_name = gc.gene_cluster_id
    JOIN kbase_ke_pangenome.gtdb_species_clade sc
        ON gc.gtdb_species_clade_id = sc.gtdb_species_clade_id
    WHERE sc.GTDB_species LIKE '%Methylobacterium%'
      AND e.Preferred_name IN ('{gene_str}')
    GROUP BY sc.GTDB_species, sc.gtdb_species_clade_id, e.Preferred_name
""")

bvit_species_pd = bvit_by_species.toPandas()

# Pivot: species x B vitamin gene presence
bvit_pivot = bvit_species_pd.pivot_table(
    index='GTDB_species', columns='Preferred_name',
    values='n_clusters', fill_value=0
)

# Merge with MDH profile
bvit_pivot = bvit_pivot.merge(
    mdh_species_pd[['GTDB_species', 'mdh_profile']].set_index('GTDB_species'),
    left_index=True, right_index=True, how='inner'
)

print(f"Species with both MDH and B vitamin data: {len(bvit_pivot)}")
print()

for profile in ['BOTH', 'xoxF_only', 'mxaF_only']:
    subset = bvit_pivot[bvit_pivot['mdh_profile'] == profile]
    if len(subset) > 0:
        numeric_cols = [c for c in subset.columns if c != 'mdh_profile']
        gene_counts = subset[numeric_cols].sum(axis=1)
        print(f"{profile} ({len(subset)} species):")
        print(f"  Mean B vitamin genes per species: {gene_counts.mean():.1f} +/- {gene_counts.std():.1f}")

## 9. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: B vitamin pathway completeness for M. extorquens
pathway_completeness = {}
for pathway, info in B_VITAMIN_PATHWAYS.items():
    expected = set(info['key_genes'])
    found = set(bvit_classified[bvit_classified['pathway'] == pathway]['Preferred_name'].unique()) & expected
    pathway_completeness[pathway.replace('_', '\n')] = len(found) / len(expected) * 100

ax1 = axes[0]
colors = ['#4CAF50' if v >= 80 else '#FF9800' if v >= 50 else '#F44336'
          for v in pathway_completeness.values()]
ax1.bar(pathway_completeness.keys(), pathway_completeness.values(), color=colors)
ax1.set_ylabel('Pathway Completeness (%)')
ax1.set_title('M. extorquens B Vitamin Biosynthesis')
ax1.set_ylim(0, 105)
ax1.axhline(80, color='gray', linestyle='--', alpha=0.5, label='80% threshold')
ax1.legend()
ax1.tick_params(axis='x', rotation=45)

# Right: B vitamin gene count by MDH profile
ax2 = axes[1]
if len(bvit_pivot) > 0:
    numeric_cols = [c for c in bvit_pivot.columns if c != 'mdh_profile']
    profile_labels = ['mxaF only', 'Both', 'xoxF only']
    profile_keys = ['mxaF_only', 'BOTH', 'xoxF_only']
    bar_colors = ['#2196F3', '#9C27B0', '#FF9800']

    for i, (profile, label) in enumerate(zip(profile_keys, profile_labels)):
        subset = bvit_pivot[bvit_pivot['mdh_profile'] == profile]
        if len(subset) > 0:
            gene_counts = subset[numeric_cols].sum(axis=1)
            ax2.bar(i, gene_counts.mean(),
                    yerr=gene_counts.std() if len(subset) > 1 else 0,
                    color=bar_colors[i], capsize=5, label=f'{label} (n={len(subset)})')
    ax2.set_xticks(range(3))
    ax2.set_xticklabels(profile_labels)
    ax2.set_ylabel('B Vitamin Genes per Species')
    ax2.set_title('MDH Profile vs B Vitamin Genes\n(Methylobacterium spp.)')
    ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../figures/bvitamin_completeness.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Core vs accessory status of B vitamin genes

Are any B vitamin biosynthesis genes in the accessory genome? If so, different *M. extorquens* strains may have different vitamin requirements.

In [ ]:
print("=== B Vitamin Gene Core/Accessory Status ===")
print()
for pathway, info in B_VITAMIN_PATHWAYS.items():
    pw_genes = bvit_classified[bvit_classified['pathway'] == pathway]
    if len(pw_genes) == 0:
        print(f"{pathway}: No genes found")
        continue

    core_genes = sorted(pw_genes[pw_genes['is_core'] == True]['Preferred_name'].unique())
    aux_genes = sorted(pw_genes[pw_genes['is_auxiliary'] == True]['Preferred_name'].unique())
    sing_genes = sorted(pw_genes[pw_genes['is_singleton'] == True]['Preferred_name'].unique())

    print(f"{pathway}:")
    if core_genes:
        print(f"  Core (all strains): {core_genes}")
    if aux_genes:
        print(f"  Auxiliary (some strains): {aux_genes}")
    if sing_genes:
        print(f"  Singleton (one strain): {sing_genes}")
    print()

print("Accessory/singleton B vitamin genes indicate strain-specific")
print("auxotrophies - different strains may need different supplements.")

## 11. Summary

### B Vitamin Biosynthesis
- _Results to be filled after execution on BERDL JupyterHub_
- Predicted auxotrophies: [TBD]
- Growth medium recommendations: [TBD]

### xoxF/mxaF and Lanthanide Biology
- eggNOG annotation caveat: xoxF is often misannotated as mxaF (EC 1.1.2.8/K00114 vs true mxaF EC 1.1.2.7/K14028)
- Cross-species MDH profile distribution: [TBD]
- B vitamin correlation with MDH profile: [TBD]

### Testable Predictions
1. *M. extorquens* AM1 growth should be enhanced by supplementing [predicted vitamins]
2. xoxF-carrying strains may have [different/similar] B vitamin requirements compared to mxaF-only strains
3. Lanthanide availability in the medium may interact with B vitamin requirements through shared cofactor demand

### Next Steps
- Validate auxotrophy predictions with growth experiments (vitamin dropout media)
- Extend analysis to other methylotrophic genera (Methylocystis, Methylomonas, Methylophaga)
- Cross-reference with environmental metadata analysis in notebooks 02-03
- Compare with GapMind pathway predictions for C1 metabolism completeness